# Hybrid Latent-State Language Model

Testing whether separating thinking from speaking improves long-horizon reasoning.

**Hypothesis:** A model with `latent_state_update() × N` + `decode_token() × M` outperforms equivalent token-by-token models.

**Experiments:**
| Exp | Model | Latent Steps |
|-----|-------|-------------|
| 001 | Transformer baseline | 0 |
| 002 | SSM only | 1 |
| 003 | SSM | 4 |
| 004 | SSM | 8 |
| 005 | SSM + FFN decoder | 4 |

In [ ]:
import sys, os, json, time
from pathlib import Path
from datetime import datetime

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from src.dataset import generate_dataset
from src.tokenizer import build_tokenizer_from_dataset
from src.models import BaselineTransformer, LatentSSM, LatentSSMDecoder
from src.trainer import Trainer, ExperimentConfig, TextDataset
from torch.utils.data import DataLoader

In [ ]:
EXPERIMENTS = [
    {"exp_id": "exp001", "model": "baseline",           "latent_steps": 0, "d_model": 256, "epochs": 30, "batch_size": 32, "n_samples": 5000},
    {"exp_id": "exp002", "model": "latent_ssm",         "latent_steps": 1, "d_model": 256, "epochs": 30, "batch_size": 32, "n_samples": 5000},
    {"exp_id": "exp003", "model": "latent_ssm",         "latent_steps": 4, "d_model": 256, "epochs": 30, "batch_size": 32, "n_samples": 5000},
    {"exp_id": "exp004", "model": "latent_ssm",         "latent_steps": 8, "d_model": 256, "epochs": 30, "batch_size": 32, "n_samples": 5000},
    {"exp_id": "exp005", "model": "latent_ssm_decoder", "latent_steps": 4, "d_model": 256, "epochs": 30, "batch_size": 32, "n_samples": 5000},
]

In [ ]:
torch.manual_seed(42)
dataset = generate_dataset(n_samples=5000, seed=42)
tokenizer = build_tokenizer_from_dataset(dataset)
print(f'Generated {len(dataset)} samples, vocab size: {tokenizer.vocab_size}')

In [ ]:
all_results = {}

for cfg in EXPERIMENTS:
    print(f"\n{'='*60}")
    print(f"{cfg['exp_id']}: {cfg['model']} (steps={cfg['latent_steps']})")
    print(f"{'='*60}")

    train_data = dataset[:4000]
    val_data = dataset[4000:]
    train_texts = [f"{s['narrative']} {s['question']} {s['answer']}" for s in train_data if s.get("question")]
    val_texts = [f"{s['narrative']} {s['question']} {s['answer']}" for s in val_data if s.get("question")]

    if cfg["model"] == "baseline":
        model = BaselineTransformer(vocab_size=tokenizer.vocab_size, d_model=cfg["d_model"], num_layers=4, nhead=8)
    elif cfg["model"] == "latent_ssm":
        model = LatentSSM(vocab_size=tokenizer.vocab_size, d_state=cfg["d_model"], d_model=cfg["d_model"], num_ssm_layers=2, latent_steps=cfg["latent_steps"])
    elif cfg["model"] == "latent_ssm_decoder":
        model = LatentSSMDecoder(vocab_size=tokenizer.vocab_size, d_state=cfg["d_model"], d_model=cfg["d_model"], num_ssm_layers=2, latent_steps=cfg["latent_steps"], tokens_per_step=8)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {n_params:,}")

    config = ExperimentConfig(exp_id=cfg["exp_id"], model=cfg["model"], d_model=cfg["d_model"], latent_steps=cfg["latent_steps"], batch_size=cfg["batch_size"], num_epochs=cfg["epochs"], save_every=5, eval_every=5)

    train_ds = TextDataset(train_texts, tokenizer, max_len=256)
    val_ds = TextDataset(val_texts, tokenizer, max_len=256)
    train_dl = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=config.batch_size)

    trainer = Trainer(model, config, tokenizer)
    metrics = trainer.train(train_dl, val_dl, qa_dataset=val_data)

    all_results[cfg["exp_id"]] = {
        "train_loss": metrics["train_loss"][-1] if metrics["train_loss"] else None,
        "val_loss": metrics["val_loss"][-1] if metrics["val_loss"] else None,
        "qa_accuracy": metrics["eval_accuracy"][-1]["overall_accuracy"] if metrics["eval_accuracy"] else None,
        "params": n_params,
    }
    torch.save(model.state_dict(), f"experiments/{cfg['exp_id']}/best_model.pt")

In [ ]:
print(f"\n{'Exp':<10} {'Model':<20} {'Steps':<6} {'Params':<10} {'Train':<10} {'Val':<10} {'QA':<6}")
print("-" * 70)
for cfg in EXPERIMENTS:
    r = all_results[cfg["exp_id"]]
    print(f"{cfg['exp_id']:<10} {cfg['model']:<20} {cfg['latent_steps']:<6} {r['params']:<10,} {r['train_loss']:<10.4f} {(r['val_loss'] or 0):<10.4f} {(r['qa_accuracy'] or 0):<6.3f}")

with open("results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nResults saved to results.json")